# 11 – Performance baseline

Measures the plan and duration of 3 queries (join, aggregation, date filter) on `trans_big` and `trans_skewed` before tuning.

Tools: `EXPLAIN` and duration. There is no Spark UI in Free Edition; Query Profile is used in `12_perf_tuning`.

This notebook writes no tables; it only reads.

## 1. Starting state

File count, size, clustering columns and row count of the three tables.

How to read: `numFiles` 1 means the data is in a single file. `clusteringColumns` shows the columns chosen automatically by serverless (empty if none).

In [0]:
from pyspark.sql import functions as F

PERF_SCHEMA = "workspace.perf"
TRANS_TABLES = ["trans_big", "trans_skewed"]
ACCOUNT_TABLE = "workspace.silver.account"

all_tables = []
for table_name in TRANS_TABLES:
    all_tables.append(f"{PERF_SCHEMA}.{table_name}")
all_tables.append(ACCOUNT_TABLE)

df_details = None
for full_name in all_tables:
    # DESCRIBE DETAIL: physical starting state of the baseline (files, size, clustering)
    df_detail = spark.sql(f"DESCRIBE DETAIL {full_name}")
    df_detail = df_detail.select(
        F.lit(full_name).alias("table_name"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
        "partitionColumns",
        "clusteringColumns",
    )
    row_count = spark.table(full_name).count()
    df_detail = df_detail.withColumn("rows", F.lit(row_count))

    if df_details is None:
        df_details = df_detail
    else:
        df_details = df_details.unionByName(df_detail)

display(df_details)

## 2. Queries and physical plan (EXPLAIN)

The three queries are defined as functions that take the trans table name as a parameter, so the same query runs on `trans_big` and `trans_skewed`.

`explain()` does not run the query; it only shows Spark's plan. The plan is the same for both tables, so only `trans_big` is shown; the effect of skew is checked with the durations.

How to read:

- Q1: `BroadcastHashJoin` (the small table is copied to every task, no shuffle) or `SortMergeJoin` (both sides are shuffled)?

- Q2: the `Exchange` / `Shuffle` line: data is redistributed by `account_id`.

- Q3: `PushedFilters` / `DataFilters` in the scan line: is the filter pushed down to the read?

- `AdaptiveSparkPlan isFinalPlan=false`: AQE can change the plan at runtime; the final plan is in Query Profile.

- Node names starting with `Photon...` mean the query runs on the Photon engine (serverless default).

In [0]:
def q1_join(trans_table):
    df_trans = spark.table(trans_table)
    # account is a small table (4,500 rows): broadcast join candidate
    df_account = spark.table(ACCOUNT_TABLE).select("account_id", "district_id")
    df_joined = df_trans.join(df_account, "account_id")
    df_result = df_joined.groupBy("district_id").agg(
        F.count("*").alias("trans_count"),
        F.sum("amount").alias("total_amount"),
    )
    return df_result


def q2_agg(trans_table):
    df_trans = spark.table(trans_table)
    # Group by account_id: in the skewed table, hot account 8261 lands in one shuffle partition
    df_result = df_trans.groupBy("account_id").agg(
        F.count("*").alias("trans_count"),
        F.sum("amount").alias("total_amount"),
    )
    return df_result


def q3_filter(trans_table):
    df_trans = spark.table(trans_table)
    # Single year (2020): shows whether the filter is pushed down to the scan
    df_filtered = df_trans.filter(
        (F.col("trans_date") >= "2020-01-01") & (F.col("trans_date") < "2021-01-01")
    )
    df_result = df_filtered.agg(
        F.count("*").alias("trans_count"),
        F.sum("amount").alias("total_amount"),
    )
    return df_result


QUERIES = {
    "q1_join": q1_join,
    "q2_agg": q2_agg,
    "q3_filter": q3_filter,
}

big_table = f"{PERF_SCHEMA}.trans_big"
for query_name in QUERIES:
    query_function = QUERIES[query_name]
    print("=" * 80)
    print(query_name)
    print("=" * 80)
    query_function(big_table).explain()

## 3. Duration

Each query runs 3 times on each table. The duration is measured with `collect()`; the results are small (at most ~4,500 rows), so pulling them to the driver is not a problem.

How to read:

- `run_no` 1 is usually slower (warm-up, cache). `min_s` is used for comparison.

- The duration includes the Spark Connect network round trip; small differences (< 0.5 s) are not meaningful.

- `result_rows`: Q1 number of districts (77), Q2 number of accounts (~4,500), Q3 a single row.

In [0]:
import time

RUNS = 3

timing_rows = []
for table_name in TRANS_TABLES:
    full_name = f"{PERF_SCHEMA}.{table_name}"
    for query_name in QUERIES:
        query_function = QUERIES[query_name]
        for run_no in range(1, RUNS + 1):
            df_query = query_function(full_name)
            # Only the collect time is measured; building the plan is not included
            start = time.perf_counter()
            result_rows = df_query.collect()
            seconds = time.perf_counter() - start
            timing_rows.append((table_name, query_name, run_no, len(result_rows), round(seconds, 2)))

df_timings = spark.createDataFrame(
    timing_rows,
    ["table_name", "query_name", "run_no", "result_rows", "seconds"],
)

df_summary = (
    df_timings.groupBy("query_name", "table_name")
    .agg(
        F.min("seconds").alias("min_s"),
        F.round(F.avg("seconds"), 2).alias("avg_s"),
        F.max("seconds").alias("max_s"),
    )
    .orderBy("query_name", "table_name")
)

display(df_timings)
display(df_summary)